## to do
* bokeh
* plot all the data, worry about averages later

* bootstrapping to get 'more' data

## 1) Get the imports

In [1]:
### All the imports
import glob, re
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import row
from bokeh.models import ColumnDataSource, Whisker
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource


## 2) Find the files, claculate means, stds

In [2]:
# 1) finding all the files
pattern = r'saltincubations_t(\d+)\.csv'
files = sorted(
    glob.glob('saltincubations_t*.csv'),
    key=lambda f: int(re.search(pattern, f).group(1))
)

weekly_stats = {}
weekly_stats_raw = {}

for fn in files:
    wk = re.search(pattern, fn).group(1)
    week_label = f't{wk}'
    
    df = pd.read_csv(fn, sep=',', thousands=',', quotechar='"', na_values=['#VALUE!'])
    df_co2 = df.iloc[:, [0,1]].copy();  df_co2.columns = ['Measured Water Activity','CO2 Conc']
    df_ch4 = df.iloc[:, [2,3]].copy();  df_ch4.columns = ['Measured Water Activity','CH4 Conc']
    
    # calc both mean and std
    co2_agg = df_co2.groupby('Measured Water Activity')['CO2 Conc'] \
                    .agg(['mean','std']).rename(columns={'mean':'mean','std':'std'})
    ch4_agg = df_ch4.groupby('Measured Water Activity')['CH4 Conc'] \
                    .agg(['mean','std']).rename(columns={'mean':'mean','std':'std'})
    
    weekly_stats[week_label] = {'co2': co2_agg, 'ch4': ch4_agg}

    # also just want the raw data for plotting later
    df_co2 = df.iloc[:, [0,1]].dropna().copy()
    df_co2.columns = ['Measured Water Activity', 'CO2 Conc']

    df_ch4 = df.iloc[:, [2,3]].dropna().copy()
    df_ch4.columns = ['Measured Water Activity', 'CH4 Conc']

    weekly_stats_raw[week_label] = {
        'co2': df_co2,
        'ch4': df_ch4
    }

# 2) cross-week summary
co2_mean_df = pd.DataFrame({wk: stats['co2']['mean'] 
                            for wk, stats in weekly_stats.items()}).T
co2_std_df  = pd.DataFrame({wk: stats['co2']['std']  
                            for wk, stats in weekly_stats.items()}).T

ch4_mean_df = pd.DataFrame({wk: stats['ch4']['mean'] 
                            for wk, stats in weekly_stats.items()}).T
ch4_std_df  = pd.DataFrame({wk: stats['ch4']['std']  
                            for wk, stats in weekly_stats.items()}).T

## 3) Output (to see trends week by week) in the first cell, and then output (to see trends depending on water activity, for every week in a separate plot). Note we're just plotting means with std

In [9]:
output_notebook()

# map week labels like 't0','t1' → numeric for x‐axis
weeks = [int(w[1:]) for w in co2_mean_df.index]
week_labelsco2 = co2_mean_df.index.tolist()
week_labelsch4 = ch4_mean_df.index.tolist()

# custom palette
palette = ['limegreen', 'green', 'teal', 'blue', 'darkblue']
def make_colors(n):
    if n <= len(palette): return palette[:n]
    return palette + ['black'] + ['grey']*(n-len(palette)-1)

colors = make_colors(len(co2_mean_df.columns))

# build two figures
p_co2 = figure(
    title="CO₂ across weeks by water activity",
    x_axis_label="Week",
    y_axis_label="Avg CO₂ Conc",
    x_range=week_labelsco2,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)
p_ch4 = figure(
    title="CH₄ across weeks by water activity",
    x_axis_label="Week",
    y_axis_label="Avg CH₄ Conc",
    x_range=week_labelsch4,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# # add lines, circles, and whiskers for each WA
for idx, wa in enumerate(co2_mean_df.columns):
    src = ColumnDataSource(data={
        'week': week_labelsco2,
        'mean':  co2_mean_df[wa].values,
        'lower': co2_mean_df[wa].values - co2_std_df[wa].values,
        'upper': co2_mean_df[wa].values + co2_std_df[wa].values
    })
    p_co2.line('week', 'mean', source=src, color=colors[idx], legend_label=f'WA {wa}')
    p_co2.scatter('week', 'mean', source=src, color=colors[idx], size=6)
    whisker = Whisker(source=src, base='week', upper='upper', lower='lower', line_color=colors[idx])
    p_co2.add_layout(whisker)

for idx, wa in enumerate(ch4_mean_df.columns):
    src = ColumnDataSource(data={
        'week': week_labelsch4,
        'mean':  ch4_mean_df[wa].values,
        'lower': ch4_mean_df[wa].values - ch4_std_df[wa].values,
        'upper': ch4_mean_df[wa].values + ch4_std_df[wa].values
    })
    p_ch4.line('week', 'mean', source=src, color=colors[idx], legend_label=f'WA {wa}')
    p_ch4.scatter('week', 'mean', source=src, color=colors[idx], size=6)
    whisker = Whisker(source=src, base='week', upper='upper', lower='lower', line_color=colors[idx])
    p_ch4.add_layout(whisker)

# style legends
p_co2.legend.title = "Water Activity"
p_co2.legend.location = "top_left"
p_ch4.legend.title = "Water Activity"
p_ch4.legend.location = "top_left"

# show side-by-side
show(row(p_co2, p_ch4))



Loading BokehJS ...

In [16]:

try:
    output_notebook()
except TypeError:
    pass

plots = []
for week_label, stats in weekly_stats.items():
    co2 = stats['co2']
    ch4 = stats['ch4']
    x = co2.index.values  # numeric water-activity levels
    
    # CO₂ source with error bars
    src_co2 = ColumnDataSource(dict(
        wa    = x,
        mean  = co2['mean'].values,
        lower = co2['mean'].values - co2['std'].values,
        upper = co2['mean'].values + co2['std'].values
    ))
    # CH₄ source with error bars
    src_ch4 = ColumnDataSource(dict(
        wa    = x,
        mean  = ch4['mean'].values,
        lower = ch4['mean'].values - ch4['std'].values,
        upper = ch4['mean'].values + ch4['std'].values
    ))
    
    # figure width and height
    p = figure(
        title=f"Week {week_label}: avg CO₂ & CH₄ ±1σ",
        x_axis_label="Measured Water Activity",
        y_axis_label="Concentration",
        tools="pan,wheel_zoom,box_zoom,reset,save",
        width=350,
        height=350
    )
    
    # CO₂ line + circles + whiskers
    p.line('wa', 'mean', source=src_co2, color='green', line_width=2, legend_label='CO₂')
    p.circle('wa', 'mean', source=src_co2, color='green', size=6)
    p.add_layout(Whisker(source=src_co2, base='wa', upper='upper', lower='lower', line_color='green'))
    
    # CH₄ line + squares + whiskers
    p.line('wa', 'mean', source=src_ch4, color='blue', line_dash='dashed', line_width=2, legend_label='CH₄')
    p.square('wa', 'mean', source=src_ch4, color='blue', size=6)
    p.add_layout(Whisker(source=src_ch4, base='wa', upper='upper', lower='lower', line_color='blue'))
    
    p.legend.title = "Gas"
    p.legend.location = "top_left"
    plots.append(p)

# arrange all the plots (one for each week) in a grid
grid = gridplot(plots, ncols=2)
show(grid)


Loading BokehJS ...

# 4) Let's now plot all the data, not just means and stds

In [5]:
# bokehhhh
try:
    output_notebook()
except TypeError:
    pass

### plot each week on a new row, with co2 in left and ch4 in right
rows = []
for week_label, raw in weekly_stats_raw.items():
    df_co2 = raw['co2']
    df_ch4 = raw['ch4']
    
    # co2 plot
    p_co2 = figure(
        title=f"Week {week_label}: Raw CO₂",
        x_axis_label="Measured Water Activity",
        y_axis_label="CO₂ Conc",
        width=350, height=350,
        tools="pan,wheel_zoom,box_zoom,reset"
    )
    src_co2 = ColumnDataSource(df_co2)
    p_co2.circle(
        'Measured Water Activity', 'CO2 Conc',
        source=src_co2,
        size=6, color='green', alpha=0.6
    )
    
    # ch4 plot
    p_ch4 = figure(
        title=f"Week {week_label}: Raw CH₄",
        x_axis_label="Measured Water Activity",
        y_axis_label="CH₄ Conc",
        width=350, height=350,
        tools="pan,wheel_zoom,box_zoom,reset"
    )
    src_ch4 = ColumnDataSource(df_ch4)
    p_ch4.square(
        'Measured Water Activity', 'CH4 Conc',
        source=src_ch4,
        size=6, color='blue', alpha=0.6
    )
    
    rows.append([p_co2, p_ch4])

# grid and then show
grid = gridplot(rows)
show(grid)


Loading BokehJS ...